# Verifying Quicopt's LABS solutions

This notebook independently checks every published solution **certificate** in
[`../solutions/labs.json`](../solutions/labs.json). For each sequence length it recomputes the
sidelobe energy directly from the bitstring and confirms that it equals both the certificate's
stated value and the published leaderboard ([`../../data/labs-v0.2.csv`](../../data/labs-v0.2.csv)).

**What a certificate is.** One bitstring per instance — the single best sequence Quicopt found.
`bits[i]='1'` means spin $s_i=+1$, `'0'` means $s_i=-1$. The objective is the off-peak
autocorrelation energy, read as a merit factor:

$$C_k \;=\; \sum_{i=0}^{N-1-k} s_i\,s_{i+k}, \qquad E \;=\; \sum_{k=1}^{N-1} C_k^2, \qquad F \;=\; \frac{N^2}{2E}.$$

**No instances to download.** Unlike Gset or MIS, a LABS instance is fully specified by its
length $N$ — there is no graph, no data file, nothing to fetch or trust. Everything below is
recomputed from the certificate itself, so this notebook runs offline and depends on nothing
in this repository except the two files it reads.

**Symmetry.** $E$ is invariant under global flip ($s\to-s$), reversal, and alternating negation
($s_i\to(-1)^i s_i$). A sequence of equal energy that looks nothing like ours is an equally
valid answer, so the test that matters is the recomputed energy — never a byte comparison
against someone else's sequence.

**Optimality.** For $N\le 66$ the optimum is *proven* by exhaustive search (Packebusch &
Mertens, [arXiv:1512.02475](https://arxiv.org/abs/1512.02475)). Those values are third-party
attributions, not Quicopt output; we check our certificates against them at the end. Above
$N=66$ no proven optimum exists and nothing here claims one.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

HERE = Path.cwd()                                   # Jupyter runs from this notebook's dir
SOLUTIONS = HERE / ".." / "solutions" / "labs.json"
LEADERBOARD = HERE / ".." / ".." / "data" / "labs-v0.2.csv"

doc = json.loads(Path(SOLUTIONS).read_text())
sols = doc["solutions"]
print(doc["solver"], "\u2014", len(sols), "certificates")
print(doc["convention"])

In [ ]:
def spins(bits):
    """s in {-1,+1}^N from the certificate bitstring: s_i = +1 iff bits[i] == '1'."""
    return np.where(np.frombuffer(bits.encode(), np.uint8) == ord("1"), 1, -1).astype(np.int64)

def sidelobe_energy(s):
    """E = sum_k C_k^2 over lags k=1..N-1, straight from the definition."""
    N = len(s)
    return int(sum(int(s[: N - k] @ s[k:]) ** 2 for k in range(1, N)))

rows = []
for inst, c in sols.items():
    s = spins(c["bits"])
    assert len(s) == c["N"], f"{inst}: bits length != N"
    assert set(c["bits"]) <= {"0", "1"}, f"{inst}: bitstring is not binary"
    E = sidelobe_energy(s)
    rows.append({"instance": inst, "N": c["N"], "claimed": c["energy"], "recomputed": E,
                 "match": E == c["energy"],
                 "F recomputed": round(c["N"] ** 2 / (2 * E), 2) if E else float("inf"),
                 "F claimed": c["merit_factor"]})

df = pd.DataFrame(rows).sort_values("N").reset_index(drop=True)

In [ ]:
# cross-check every certificate against the published leaderboard row for the same instance
lb = pd.read_csv(LEADERBOARD).set_index("instance")
df["leaderboard"] = df["instance"].map(lb["energy"])
df["== leaderboard"] = df["claimed"] == df["leaderboard"]

# %_of_best is the ratio of merit factors, E_ref / E, so it must follow from the two columns
# it is derived from. Recompute it rather than trust it.
df["best-known"] = df["instance"].map(lb["best-known"])
df["%_of_best"] = df["instance"].map(lb["%_of_best"])
df["%_recomputed"] = (100.0 * df["best-known"] / df["recomputed"]).round(2)

print(f"verified {len(df)} instances, N = {df['N'].min()}–{df['N'].max()}")
print("all recomputed energies == certificate:", bool(df["match"].all()))
print("all merit factors consistent:          ", bool((df["F recomputed"] == df["F claimed"]).all()))
print("all certificates == leaderboard:       ", bool(df["== leaderboard"].all()))
print("all %_of_best follow from the columns:  ", bool((df["%_recomputed"] == df["%_of_best"]).all()))
print("never better than the reference:       ", bool((df["recomputed"] >= df["best-known"]).all()))
assert df["match"].all(), "a recomputed energy disagrees with its certificate!"
assert (df["F recomputed"] == df["F claimed"]).all(), "a merit factor disagrees with its energy!"
assert df["== leaderboard"].all(), "a certificate disagrees with the published leaderboard!"
assert (df["%_recomputed"] == df["%_of_best"]).all(), "a %_of_best disagrees with its own inputs!"
assert (df["recomputed"] >= df["best-known"]).all(), "a certificate beats the published reference!"
df

### Against the proven optima, N ≤ 66

Exhaustive-search optima from Packebusch & Mertens ([arXiv:1512.02475](https://arxiv.org/abs/1512.02475)),
indexed by sequence length. These are third-party values, reproduced here only so the
certificates can be graded against them; above $N=66$ the table ends and no optimality claim is
made anywhere in this repository.

Their table starts at $N=3$, so 64 lengths are graded below. The leaderboard's `source` column
marks 65 rows `optimum`: the extra one is $N=2$, whose optimum $E=1$ is immediate and needs no
search.

In [ ]:
PROVEN_OPTIMA = {n: e for n, e in enumerate([
    0, 0, 0, 1, 2, 2, 7, 3, 8, 12, 13, 5, 10, 6, 19, 15, 24, 32, 25, 29, 26, 26, 39, 47,
    36, 36, 45, 37, 50, 62, 59, 67, 64, 64, 65, 73, 82, 86, 87, 99, 108, 108, 101, 109,
    122, 118, 131, 135, 140, 136, 153, 153, 166, 170, 175, 171, 192, 188, 197, 205, 218,
    226, 235, 207, 208, 240, 257]) if n >= 3}

graded = df[df["N"].isin(PROVEN_OPTIMA)].copy()
graded["optimum"] = graded["N"].map(PROVEN_OPTIMA)
graded["gap"] = graded["recomputed"] - graded["optimum"]

reached = int((graded["gap"] == 0).sum())
print(f"{reached}/{len(graded)} proven optima reached")
print("never below a proven optimum:", bool((graded["gap"] >= 0).all()))
assert (graded["gap"] >= 0).all(), "a certificate beats a proven optimum \u2014 recheck the energy!"
graded[["instance", "N", "recomputed", "optimum", "gap"]]